In [ ]:
import os
import torch
from functools import partial
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

from spikingjelly.activation_based import neuron, surrogate, functional, monitor

from src.utils.data_utils import get_dataloaders
from src.utils.model_utils import get_model, get_conv_layers, get_lif_layers
from src.utils.save_utils import load_model
from src.utils.plot_utils import plot_frames, plot_cam_time_series, combine_polarities, plot_aggregated_cam
from src.cams.generation_utils import (
    generate_grad_cam, 
    generate_ablation_cam, 
    generate_mc_ablation_cam, 
    generate_sam,
    generate_mc_ablation_cam_transient,
    generate_noise_grad_cam,
    generate_guided_backprop_cam,
    generate_replace_surrogate_grad_cam,
    generate_smooth_grad_cam,
    generate_grad3d,
    generate_integrated_gradients,
    generate_smooth_noise_grad_cam
)

# Repro + device
print(torch.seed())
# torch.manual_seed(16545676774062543603)

# Configuration
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATASET_NAME = "dvs_gesture"  # Change as needed
MODEL_NAME = "spiking_vgg11_bn"     # Change as needed
MODEL_DIR = "./models_save"   # Directory where models are saved
DATA_DIR = "./data"
N_TIME_BINS = 16
TAU=2.0

print(f"Running on {DEVICE}")

#9310193271258541288

In [ ]:
_, test_loader = get_dataloaders(DATASET_NAME, DATA_DIR, batch_size=1, n_time_bins=N_TIME_BINS)
sample_events, sample_labels = next(iter(test_loader))

was_single_image = False

if sample_events.dim() == 4:
    was_single_image = True
    sample_events = sample_events.unsqueeze(1).repeat(1, N_TIME_BINS, 1, 1, 1)
sample_events = sample_events.transpose(0, 1)
T, B, C, H, W = sample_events.shape
print('Events Shape (T, B, C, H, W):', (T, B, C, H, W))

if sample_labels.dim() == 2:
    num_classes = sample_labels.size(1)
else:
    ds = test_loader.dataset
    if hasattr(ds, "classes"):
        num_classes = len(ds.classes)
    elif hasattr(ds, "dataset") and hasattr(ds.dataset, "classes"):
        num_classes = len(ds.dataset.classes)
    else:
        raise ValueError("Could not determine number of classes from dataset.")
    
spiking_neuron = neuron.LIFNode if not was_single_image else neuron.IFNode

neuron_arguments = {
    "tau": TAU,
    "detach_reset": True,
    "surrogate_function": surrogate.ATan()
}

if spiking_neuron == neuron.IFNode:
    neuron_arguments.pop("tau")

print("The spiking neuron used is:", spiking_neuron)

params = {
    "model_name": MODEL_NAME,
    "num_classes": num_classes,
    "in_channels": C,
    "spiking_neuron": spiking_neuron,
    "pretrained": False,
}

model = get_model(**params)

functional.set_step_mode(model, 'm')
functional.set_backend(model, "torch", instance=spiking_neuron)
model.to(DEVICE)

load_model(model, MODEL_DIR, DATASET_NAME)
model.eval()
print("Model loaded.")

gesture_map = {
    0: "hand_clapping", 1: "right_hand_wave", 2: "left_hand_wave", 3: "right_hand_clockwise",
    4: "right_hand_counter_clockwise", 5: "left_hand_clockwise", 6: "left_hand_counter_clockwise",
    7: "forearm_roll_forward", 8: "forearm_roll_backward", 9: "drums", 10: "guitar",
}

In [ ]:
def benchmark_sparsity(net, dataloader, device='cuda', ignore_final_layer=False):
    net.eval()
    net.to(device)
    
    # We will track the total number of spikes and total neuron-steps across the whole dataset
    total_spikes = 0
    total_elements = 0
    
    # 1. Register a monitor to record output spikes of every spiking neuron layer
    #    OutputMonitor automatically records the output of layers you specify
    spike_monitor = monitor.OutputMonitor(net, neuron.LIFNode)
    
    try:
        with torch.no_grad():
            for img, label in dataloader:
                if img.dim() == 4:
                    img = img.unsqueeze(1)  # add time dimension
                    img = img.repeat(1, N_TIME_BINS, 1, 1, 1)  # repeat along time dimension
        
                img = img.transpose(0, 1).to(DEVICE)

                img = img.to(DEVICE)
                
                # Forward pass
                functional.reset_net(net)
                
                # If your input is static, you might repeat it over T timesteps here
                # For this snippet, assuming img is already (T, B, C, H, W) or handled by the net
                net(img)
                
                # 2. Iterate through recorded spike tensors to calculate sparsity
                # spike_monitor.records is a list of tensors, one for each monitored layer
                records = spike_monitor.records
                if ignore_final_layer and len(records) > 0:
                    records = records[:-1]

                for spike_tensor in records:
                    # spike_tensor shape: [T, Batch, ...]
                    total_spikes += spike_tensor.sum().item()
                    total_elements += spike_tensor.numel()
                spike_monitor.clear_recorded_data()

    finally:
        spike_monitor.remove_hooks()
        del spike_monitor
        # 3. Calculate Global Average Firing Rate
        avg_firing_rate = total_spikes / total_elements
        print(f"Global Average Firing Rate (Sparsity): {avg_firing_rate:.4f}")
        

    return avg_firing_rate

def print_memory_usage(device):
    free, total = torch.cuda.mem_get_info(device)
    mem_used_MB = (total - free) / 1024 ** 2
    print(f"Memory Used: {mem_used_MB:.2f} MB")

def print_cuda_memory_breakdown(local_vars, top_k=10):
    print(f"\n--- Top {top_k} CUDA Memory Consumers ---")
    
    # Get current memory stats
    allocated = torch.cuda.memory_allocated(DEVICE) / (1024 ** 2)
    reserved = torch.cuda.memory_reserved(DEVICE) / (1024 ** 2)
    print(f"Total Allocated: {allocated:.2f} MB")
    print(f"Total Reserved: {reserved:.2f} MB")
    print(f"Difference (fragmentation/cache): {reserved - allocated:.2f} MB\n")
    
    cuda_vars = []
    
    for var_name, var_val in local_vars.items():
        # Check for direct Tensor
        if torch.is_tensor(var_val) and var_val.is_cuda:
            mem = var_val.element_size() * var_val.nelement()
            cuda_vars.append((var_name, mem, var_val.shape, "Tensor"))
        
        # Check for lists/tuples of Tensors
        elif isinstance(var_val, (list, tuple)) and len(var_val) > 0:
            total_mem = 0
            count = 0
            for item in var_val:
                if torch.is_tensor(item) and item.is_cuda:
                    total_mem += item.element_size() * item.nelement()
                    count += 1
            if count > 0:
                cuda_vars.append((var_name, total_mem, f"len={len(var_val)}, tensors={count}", "List/Tuple"))
        
        # Check for dict of Tensors
        elif isinstance(var_val, dict):
            total_mem = 0
            count = 0
            for v in var_val.values():
                if torch.is_tensor(v) and v.is_cuda:
                    total_mem += v.element_size() * v.nelement()
                    count += 1
            if count > 0:
                cuda_vars.append((var_name, total_mem, f"keys={len(var_val)}, tensors={count}", "Dict"))

    # Sort by memory usage (descending)
    cuda_vars.sort(key=lambda x: x[1], reverse=True)
    
    total_tracked = 0
    for name, mem_bytes, shape, var_type in cuda_vars[:top_k]:
        mem_mb = mem_bytes / (1024 ** 2)
        total_tracked += mem_mb
        print(f"{name} ({var_type}): {mem_mb:.2f} MB | Info: {shape}")
    
    print(f"\nTotal Tracked: {total_tracked:.2f} MB")
    print(f"Untracked (cache/fragmentation/gradients): {allocated - total_tracked:.2f} MB")
    print("-----------------------------------------")

# avg_firing_rate = benchmark_sparsity(model, test_loader, device=DEVICE, ignore_final_layer=True)

In [ ]:
import time


K_TRIALS = 200
CHANNEL_BATCH_SIZE = 4

def cam_variance_T_HW(cam):
    if isinstance(cam, torch.Tensor):
        cam = cam.detach().cpu().numpy()
    return float(cam.var(axis=(1, 2)).mean())

conv_layers = get_conv_layers(model)
lif_layers = get_lif_layers(model)
print(f'Found {len(conv_layers)} conv layers and {len(lif_layers)} (L)IF layers.')
target_layer_indices = [3, 5, 7]

for events, labels in test_loader:
    functional.reset_net(model)
    was_single_image = False
    if events.dim() == 4:
        was_single_image = True
        events = events.unsqueeze(1).repeat(1, N_TIME_BINS, 1, 1, 1)
    
    pre_encoding_events = events.clone()
    events = events.transpose(0, 1).to(DEVICE)
    label = labels.to(DEVICE)
    
    target_classes = []
    if label.dim() == 2 and label.shape[1] > 1:
        target_classes = torch.nonzero(label[0]).flatten().tolist()
    else:
        target_classes = [int(label[0].item())]

    print(f'\nGround Truth Label(s): {target_classes}')
    if "dvs_gesture" in DATASET_NAME:
        print(f"Gesture Name(s): {[gesture_map.get(c, 'Unknown') for c in target_classes]}")
    
    outputs = model(events)
    time_sum = outputs.sum(dim=0)
    
    top_preds = time_sum[0].topk(min(5, num_classes)).indices.tolist()
    valid_classes = [t for t in target_classes if t in top_preds]
    
    if not valid_classes:
        print(f"Skipping CAM generation. No GT classes {target_classes} found in top predictions {top_preds}.")
        continue

    print(f"Generating CAMs for correctly identified classes: {valid_classes}")

    frames = pre_encoding_events[0, :].detach().cpu()
    if frames.shape[1] == 2:
        combined = combine_polarities(frames.permute(0,2,3,1).numpy())
        combined = np.expand_dims(combined, axis=1)
    else:
        combined = frames.numpy()

    plot_frames(combined)

    try:
        for target_class in valid_classes:
            print(f"\n=== Generating CAMs for Class {target_class} ===")
            for layer_index in target_layer_indices:
                if layer_index >= len(conv_layers) or layer_index >= len(lif_layers):
                    break
                print(f'Targeting layer index {layer_index}')
                
                # Grad-CAM
                grad_cams = generate_grad_cam(events, target_class, layer_index, model)
                print("Grad-CAMs over time:")
                plot_cam_time_series(combined, grad_cams, images_per_row=8)
                plot_aggregated_cam(combined, grad_cams)
                del grad_cams
                torch.cuda.empty_cache()

                # Noise Grad-CAM
                start_time = time.perf_counter()
                noise_grad_cams = generate_noise_grad_cam(events, target_class, layer_index, model, k_trials=K_TRIALS, batch_size=CHANNEL_BATCH_SIZE, noise_type='gaussian')
                end_time = time.perf_counter()
                print(f"Noise Grad-CAM generation took {end_time - start_time:.2f} seconds, {(end_time - start_time) / K_TRIALS:.2f} seconds per trial.")
                print("Noise Grad-CAMs over time:")
                plot_cam_time_series(combined, noise_grad_cams, images_per_row=8)
                plot_aggregated_cam(combined, noise_grad_cams)
                del noise_grad_cams
                torch.cuda.empty_cache()
                
                start_time = time.perf_counter()
                smooth_grad_cams = generate_smooth_grad_cam(events, target_class, layer_index, model, n_samples=K_TRIALS, stdev_spread=0.15)
                end_time = time.perf_counter()
                print(f"SmoothGrad-CAM generation took {end_time - start_time:.2f} seconds, {(end_time - start_time) / K_TRIALS:.2f} seconds per trial.")
                print("SmoothGrad-CAMs over time:")
                plot_cam_time_series(combined, smooth_grad_cams, images_per_row=8)
                plot_aggregated_cam(combined, smooth_grad_cams)
                del smooth_grad_cams
                torch.cuda.empty_cache()

                start_time = time.perf_counter()
                smooth_noise_grad_cams = generate_smooth_noise_grad_cam(events, target_class, layer_index, model, k_trials=K_TRIALS, batch_size=CHANNEL_BATCH_SIZE, stdev_spread=0.15, noise_type='uniform')
                end_time = time.perf_counter()
                print(f"Smooth Noise Grad-CAM generation took {end_time - start_time:.2f} seconds, {(end_time - start_time) / K_TRIALS:.2f} seconds per trial.")
                print("Smooth Noise Grad-CAMs over time:")
                plot_cam_time_series(combined, smooth_noise_grad_cams, images_per_row=8)
                plot_aggregated_cam(combined, smooth_noise_grad_cams)
                del smooth_noise_grad_cams
                torch.cuda.empty_cache()

                # # Ablation CAM
                ab_cams = generate_ablation_cam(events, target_class, layer_index, model, batch_size=CHANNEL_BATCH_SIZE)
                print("Ablation CAMs over time:")
                plot_cam_time_series(combined, ab_cams, images_per_row=8)
                plot_aggregated_cam(combined, ab_cams)

                # SAM
                sams = generate_sam(events, layer_index, model)
                print("SAMs over time:")
                plot_cam_time_series(combined, sams, images_per_row=8)
                plot_aggregated_cam(combined, sams)
    
    except RuntimeError as e:
        print(f"Caught RuntimeError during CAM generation: {e}")
        print_cuda_memory_breakdown(locals(), top_k=10)
        raise e
    
    break  # Process only one batch for demonstration

In [ ]:
# paste back in when needed, here to not bloat
                # # Noise Grad-CAM
                # noise_grad_cams = generate_noise_grad_cam(events, target_class, layer_index, model, k_trials=100, batch_size=CHANNEL_BATCH_SIZE)
                # print("Noise Grad-CAMs over time:")
                # plot_cam_time_series(combined, noise_grad_cams, images_per_row=8)
    
                # del noise_grad_cams
                # torch.cuda.empty_cache()    

                # # Noise Grad-CAM uniform noise
                # noise_grad_cams = generate_noise_grad_cam(events, target_class, layer_index, model, k_trials=100, batch_size=CHANNEL_BATCH_SIZE, noise_type="uniform")
                # print("Noise Grad-CAMs (uniform) over time:")
                # plot_cam_time_series(combined, noise_grad_cams, images_per_row=8)

               # print_memory_usage(DEVICE)

                # # MC Ablation CAM
                # mc_ablation_cams, _ = generate_mc_ablation_cam(events, target_class, layer_index, model, k_trials=K_TRIALS, batch_size=CHANNEL_BATCH_SIZE)
                # print("MC Ablation CAMs over time:")
                # plot_cam_time_series(combined, mc_ablation_cams, images_per_row=8)

                # del mc_ablation_cams, _
                # torch.cuda.empty_cache()

                # mode = "3d" if not was_single_image else "2d"
                # integrated_gradients = generate_integrated_gradients(events, target_class, model, steps=50)
                # print("Integrated Gradients over time:")
                # plot_cam_time_series(combined, integrated_gradients, images_per_row=8)
                # del integrated_gradients
                # torch.cuda.empty_cache()

                # grad3d_cams = generate_grad3d(events, target_class, model)
                # print("Grad3D CAMs over time:")
                # plot_cam_time_series(combined, grad3d_cams, images_per_row=8)
                # del grad3d_cams
                # torch.cuda.empty_cache()

                # # MC Ablation CAM Transient
                # start_time = time.perf_counter()
                # mc_ablation_cams_transient, _ = generate_mc_ablation_cam_transient(events, target_class, layer_index, model, k_trials=K_TRIALS, batch_size=CHANNEL_BATCH_SIZE, noise_type="gaussian")
                # end_time = time.perf_counter()
                # print(f"MC Ablation CAM Transient generation took {end_time - start_time:.2f} seconds, {(end_time - start_time) / K_TRIALS:.2f} seconds per trial.")
                # print("MC Ablation CAM Transient over time:")
                # plot_cam_time_series(combined, mc_ablation_cams_transient, images_per_row=8)

                # del mc_ablation_cams_transient, _
                # torch.cuda.empty_cache()

                # # Noise Grad-CAM
                # noise_grad_cams = generate_noise_grad_cam(events, target_class, layer_index, model, k_trials=100, batch_size=CHANNEL_BATCH_SIZE)
                # print("Noise Grad-CAMs over time:")
                # plot_cam_time_series(combined, noise_grad_cams, images_per_row=8)

                # # Replace Surrogate Grad-CAM
                # replace_surrogate_grad_cams = generate_replace_surrogate_grad_cam(events, target_class, layer_index, model)
                # print("Replace Surrogate Grad-CAMs over time: (Using LeakyKReLU with leak=0, k=1)")
                # plot_cam_time_series(combined, replace_surrogate_grad_cams, images_per_row=8)

                # # Replace Surrogate Grad-CAM non-spiking replacement
                # new_surrogate = surrogate.ATan(spiking=False)
                # replace_surrogate_grad_cams_non_spiking = generate_replace_surrogate_grad_cam(events, target_class, layer_index, model, new_surrogate=new_surrogate)
                # print("Replace Surrogate Grad-CAMs over time: (Using ATan non-spiking surrogate)")
                # plot_cam_time_series(combined, replace_surrogate_grad_cams_non_spiking, images_per_row=8)

                # # Replace surrogate gradient with a tighter approximation
                # new_surrogate_tighter = surrogate.ATan(alpha=5.0, spiking=True)
                # replace_surrogate_grad_cams_tighter = generate_replace_surrogate_grad_cam(events, target_class, layer_index, model, new_surrogate=new_surrogate_tighter)
                # print("Replace Surrogate Grad-CAMs over time: (Using ATan spiking surrogate with alpha=5.0)")
                # plot_cam_time_series(combined, replace_surrogate_grad_cams_tighter, images_per_row=8)

                # # Replace the surrogate gradient with an even tighter approximation
                # new_surrogate_even_tighter = surrogate.ATan(alpha=10.0, spiking=True)
                # replace_surrogate_grad_cams_even_tighter = generate_replace_surrogate_grad_cam(events, target_class, layer_index, model, new_surrogate=new_surrogate_even_tighter) 
                # print("Replace Surrogate Grad-CAMs over time: (Using ATan spiking surrogate with alpha=10.0)")
                # plot_cam_time_series(combined, replace_surrogate_grad_cams_even_tighter, images_per_row=8)